In [1]:
import numpy as np
from matplotlib import pyplot as plt
import scipy as sp
from scipy.optimize import bisect
from scipy.integrate import quad

import os
from matplotlib import rc
rc('text', usetex=True)
from matplotlib.lines import Line2D

In [2]:
#Units
eV = 1
GeV = 1e9 * eV
MeV = 1e3 * eV
Kel = 8.617333262 * 1e-5 * eV

#SM constants
T0 = 2.725 * Kel #in eV.  #CMB temp today
me = 0.000511 * GeV
mp = 0.938 * GeV
alpha = 0.0073

#This code works with everything in eV, so we need to convert H into those units. 
hbar = 6.582119 * 10**-16 #(eV s)
c = 299792458 # m/s
invMpc_to_eV = (1/(3.086 * 10**22)) * hbar * c #If you have a number in inverse megaparsecs, multiply by this to get the number in eV


#Optional: You can do this or get the abundances from CLASS. 
# #Get a table of H(z) values from a CLASS run. 
# background_arr = np.loadtxt('class_adm-3.1/output/for_spectral_distortions/f0.01_deltaN0.001_mp0.01GeV_me0.1MeV_alpha0.05_01_background.dat')
# H_arr = background_arr[:,3] #In 1/Mpc
# z_arr = background_arr[:,0]

# def H_class(z):#Returns H in eV units, interpolated.  
#     return np.interp(z,np.flip(z_arr),np.flip(H_arr)) * invMpc_to_eV

#Hubble constant. 
#Set h = 0.67, and then H0 = h * invMpc_to_eV * 1000 * 100 / c
#If you want to use output from CLASS, you can pull it from CLASS background.dat output. 

h =  0.67 #H_arr[-1] * c / 1000 /100 #H0 in km/s/Mpc divided by 100. 
H0 = h * invMpc_to_eV * 100 * 1000 / c #(H_arr[-1] * invMpc_to_eV) #H0 in eV. 

#aDM parameters. Fraction of DM, DeltaNeff, temperature ratio with CMB today, dark proton mass, dark electron mass, dark fine structure constant, dark reduced mass, dark hydrogen binding energy, dark CMB temperature today, kinetic mixing parameter. 
# fd = 0.01
# DeltaN = 0.001
# xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
# mpd = 0.01 * GeV
# med = 0.0001 * GeV
# alphad = 0.05
# mu = mpd * med/(mpd + med) 
# Bd = alphad**2 * mu/2
# Td0 = xid * T0
# epsilon = 1e-7

#If you want to use the dark electron ionization fraction from CLASS for some reason, this loads that array in. There's functions that are defined later to use the CLASS ionization fraction . 
# try:
#     #Load the dark electron ionization fraction tabulated for many values of z. 
#     xed_class_arr = np.loadtxt('class_adm-3.1/output/for_spectral_distortions/f%g_deltaN%g_mp%gGeV_me%gMeV_alpha%g_ionization.txt'%(fd,DeltaN,mpd/GeV,med*MeV/GeV,alphad))
#     print('Dark electron ionization curve succesfully loaded from CLASS output')
# except FileNotFoundError:
#     print('No CLASS output file found for these parameters')
#     pass
#Interpolate the dark electron ionization fraction.  
def xed_class(z):
    return np.interp(z,xed_class_arr[:,0],xed_class_arr[:,1])

#Interpolate to get the derivative of the dark electron ionization fraction. 
def dxed_class(z):
    deps = 0.001
    rise = np.interp(z*(1+deps),xed_class_arr[:,0],xed_class_arr[:,1]) - np.interp(z*(1-deps),xed_class_arr[:,0],xed_class_arr[:,1])
    run = 2 * deps * z
    return rise/run


# G = 6.6743 * 10**-11 #Newton's gravitational constant in SI. #meters ^3 / kg / s^2 = meters^5 seconds ^-4 / J. 
# G *= 1.60218*10**-19 #now in meters^5 seconds^-4 / eV
# G *= (1/c)**5 #now in seconds / eV
# G *= (1/hbar) #now in 1/eV**2
# print(G)

G = 6.708845421233845e-57 #In units of 1/eV^2

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
#Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
#nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)

#Hubble. Could use interpolated table from CLASS if you want. 
def H(z):
    return np.sqrt(H0**2 * (Omegar*(1+z)**4 + Omegam*(1+z)**3 + OmegaLambda))

#Hydrogen number density at z
def n(z):
    return n0 * (1+z)**3

#Derivative of hydrogen number density at z
def dn(z):
    return 3 * n0 * (1+z)**2

#CMB temperature at z
def Tcmb(z):
    return T0 * (1+z)

#Dark hydrogen number density at z (overall density, not dependent on ionization fraction)
def nd(z):
    return nd0 * (1+z)**3

#Derivative of dark hydrogen number density at z
def dnd(z):
    return 3 * nd0 * (1+z)**2

#Dark CMB temperature at z
def Td(z):
    return Td0 * (1+z)

#Derivative of dark CMB temperature at z
def dTd(z):
    return Td0

#Right-hand-side of Saha equation at z, equal to (1-xed)/xed^2
def RHS(z):
    return (1/nd(z)) * (Td(z) * med / (2 * np.pi))**(3/2) * np.exp(-Bd/Td(z),dtype=np.longdouble)

#Derivative of right-hand-side of Saha equation
def dRHS(z):
    return (Bd/(Td0 * (1+z)**2) - 3/(2 * (1+z))) * RHS(z)

#Dark electron ionization fraction at z
def xed(z):
    return np.where(z < 0.2*Bd/Td0, (1/2) * (-RHS(z) + np.sqrt(RHS(z)) * np.sqrt(4 + RHS(z),dtype=np.longdouble)),1.0)
    
#Derivative of dark electron ionization fraction at z
def dxed(z):
    u = RHS(z)
    return (1/2)*(-1 + np.sqrt(u,dtype=np.longdouble)/(2 * np.sqrt(4 + u,dtype=np.longdouble)) + np.sqrt(4 + u,dtype=np.longdouble)/(2*np.sqrt(u,dtype=np.longdouble))) * dRHS(z)

#Dark photon mass squared at z and frequency xd
def mgd2(z,xd):
    return (4 * np.pi * alphad / med) * xed(z) * nd(z) * (1 + (med/mpd) - (1/200.0) * xd**2 * (Td(z)/eV)**2 * (1 - xed(z))/xed(z))

#Derivative of dark photon mass squared at z and frequency xd
def dmgd2(z,xd):
    term1 = dxed(z)/xed(z)
    term2 = dnd(z)/nd(z)
    num3 = -(1/200.0) * xd**2 * (2 * (Td(z)/eV)*(dTd(z)/eV) * ((1-xed(z))/xed(z)) + (Td(z)/eV)**2 * (-1/xed(z)**2)*dxed(z))
    denom3 = (1 + (med/mpd) - (1/200.0) * xd**2 * (Td(z)/eV)**2 * (1 - xed(z))/xed(z))
    term3 = num3/denom3
    return mgd2(z,xd) * (term1 + term2 + term3)

#Photon mass squared at z: Omits frequency dependence. 
def mg2(z):
    return 4 * np.pi * alpha * n(z) / me

#Derivative of photon mass squared at z
def dmg2(z):
    return 4 * np.pi * alpha * dn(z) / me

#Photon to dark photon conversion factor. Only valid at z_resonance. 
def gammacon(xd): 
    z = find_zres(xd)
    return (np.pi * epsilon**2 * mg2(z))/(3 * Tcmb(z) * H(z))/np.abs(1 - ((1 + z)/(3 * mg2(z))) * dmgd2(z,xd))

#Dark photon mass squared, using dark electron ionization fraction from CLASS instead of Saha equation. 
def mgd2_class(z,xd):
    xed = xed_class(z)
    return (4 * np.pi * alphad / med) * xed * nd(z) * (1 + (med/mpd) - (1/200.0) * xd**2 * (Td(z)/eV)**2 * (1 - xed)/xed)

#Derivative of dark photon mass squared, using dark electron ionization fraction from CLASS instead of Saha equation. 
def dmgd2_class(z,xd):

    xed = xed_class(z)
    term1 = dxed(z)/xed
    term2 = dnd(z)/nd(z)
    num3 = -(1/200.0) * xd**2 * (2 * (Td(z)/eV)*(dTd(z)/eV) * ((1-xed)/xed) + (Td(z)/eV)**2 * (-1/xed**2)*dxed_class(z))
    denom3 = (1 + (med/mpd) - (1/200.0) * xd**2 * (Td(z)/eV)**2 * (1 - xed)/xed)
    term3 = num3/denom3
    return mgd2(z,xd) * (term1 + term2 + term3)
    
#Photon to dark photon conversion factor, using dark electron ionization fraction from CLASS instead of Saha equation. Only valid at z_resonance. 
def gammacon_class(xd): #Only valid at z_resonance. 
    z = find_zres_class(xd)
    return (np.pi * epsilon**2 * mg2(z))/(3 * Tcmb(z) * H(z))/np.abs(1 - ((1 + z)/(3 * mg2(z))) * dmgd2_class(z,xd))

#Difference between dark photon mass squared and photon mass squared 
def mg_wrapper(logz,xd):
    z = np.exp(logz,dtype=np.longdouble)
    return mgd2(z,xd)- mg2(z)

#Function that uses bisection method to find the z where the dark photon mass and photon mass are equal. 
def find_zres(xd):
    return np.exp(bisect(mg_wrapper,np.log(0.01 * Bd/Td0),np.log(0.2 * Bd/Td0),args=(xd),disp=True),dtype=np.longdouble)

#Difference between dark photon mass squared and photon mass squared, using dark electron ionization fraction from CLASS instead of Saha equation. 
def mg_wrapper_class(logz,xd):
    z = np.exp(logz,dtype=np.longdouble)
    return mgd2_class(z,xd)- mg2(z)
#Function that uses bisection method to find the z where the dark photon mass and photon mass are equal, using dark electron ionization fraction from CLASS instead of Saha equation.
def find_zres_class(xd):
    return np.exp(bisect(mg_wrapper_class,np.log(0.01 * Bd/Td0),np.log(0.2 * Bd/Td0),args=(xd),disp=True),dtype=np.longdouble)

#Numerical factors. Gn = int_0^inf x^n / (e^x -1) dx
G1 = 1.64493
G2 = 2.40411
G3 = 6.49394

#Integrand that appears in eq. 1.46, to give Deltarho/rho
def rho_integrand(x):
    return x**2 * (1/(np.exp(x) - 1) - 1/(np.exp(x/xid) - 1)) * gammacon(x/xid)


#Integrand that appears in eq. 1.48, to give DeltaN/n
def N_integrand(x):
    return x * (1/(np.exp(x) - 1) - 1/(np.exp(x/xid) - 1)) * gammacon(x/xid)

def set_global_params(params):
    global fd
    global DeltaN
    global mpd
    global med
    global alphad
    global xid
    global mu
    global Bd
    global Td0
    global Omegar
    global nd0
    
    fd = params[0]
    DeltaN = params[1]
    mpd = params[2]
    med = params[3]
    alphad = params[4]
    xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
    mu = mpd * med/(mpd + med)
    Bd = alphad**2 * mu/2
    Td0 = xid * T0
    
    z_recomb_estimate = 0.01 * Bd/Td0
    
    Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.    
    #Number density of dark hydrogen today
    nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)

def get_mu_distortion(params):
    global fd
    global DeltaN
    global mpd
    global med
    global alphad
    global xid
    global mu
    global Bd
    global Td0
    global Omegar
    global nd0
    
    fd = params[0]
    DeltaN = params[1]
    mpd = params[2]
    med = params[3]
    alphad = params[4]
    xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
    mu = mpd * med/(mpd + med)
    Bd = alphad**2 * mu/2
    Td0 = xid * T0
    
    z_recomb_estimate = 0.01 * Bd/Td0
    
    Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.    
    #Number density of dark hydrogen today
    nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
    
    if (2e3 < z_recomb_estimate) and (z_recomb_estimate < 1e8):
        if mgd2(1e8,1) > mg2(1e8):    
            rho_integral = quad(rho_integrand,0,50)[0]
            N_integral = quad(N_integrand,0,50)[0]
            Deltarho_rho = (-1/G3)*rho_integral
            DeltaN_N = (-1/G2) * N_integral
            mu_distortion = 1.401 * (Deltarho_rho - (4/3) * DeltaN_N)
            return mu_distortion 
        else:
            print('Dark photon mass is not larger than photon mass at high redshift')
            return -1
    else:
        print('Dark hydrogen recombination happens before z=1e8 or after z=1e4')
        return -1

def dlnmgddlnz(z,xd):
    return z/(2 * mgd2(z,xd)) * dmgd2(z,xd)

In [3]:
#Let's explicitly test the dependence of the mu distortion on z_res vs dmgd2/dz(z_res). 

def get_mu_distortion_plus(params):
    global fd
    global DeltaN
    global mpd
    global med
    global alphad
    global xid
    global mu
    global Bd
    global Td0
    global Omegar
    global nd0
    
    fd = params[0]
    DeltaN = params[1]
    mpd = params[2]
    med = params[3]
    alphad = params[4]
    xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4) #ratio of dark temp to photon temp
    mu = mpd * med/(mpd + med) #reduced mass of dark hydrogen
    Bd = alphad**2 * mu/2 
    Td0 = xid * T0
    
    z_recomb_estimate = 0.01 * Bd/Td0
    
    Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.    
    #Number density of dark hydrogen today
    nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)

    
    zres1 = find_zres(1)
    zres2 = find_zres(0.1)
    zres3 = find_zres(0.01)

    deriv1 = dmgd2(zres1,1)
    deriv2 = dmgd2(zres2,0.1)
    deriv3 = dmgd2(zres3,0.01)

    D1 = dlnmgddlnz(zres1,1)
    D2 = dlnmgddlnz(zres2,0.1)
    D3 = dlnmgddlnz(zres3,0.01)
    
    if (2e3 < z_recomb_estimate) and (z_recomb_estimate < 1e8):
        if mgd2(1e8,1) > mg2(1e8):    
            rho_integral = quad(rho_integrand,0,50)[0]
            N_integral = quad(N_integrand,0,50)[0]
            Deltarho_rho = (-1/G3)*rho_integral
            DeltaN_N = (-1/G2) * N_integral
            mu_distortion = 1.401 * (Deltarho_rho - (4/3) * DeltaN_N)
            return (mu_distortion,zres1,zres2,zres3,deriv1,deriv2,deriv3,D1,D2,D3)
        else:
            print('Dark photon mass is not larger than photon mass at high redshift')
            return -1
    else:
        print('Dark hydrogen recombination happens before z=1e8 or after z=1e4')
        return -1

#We want to calculate the mu distortion, manually setting the redshift of the resonance as well as the derivative of mgamma^2 w.r.t z. 
#How does dmgamma^2/dz naturally scale with z_res? 


fd = 0.01
epsilon = 1e-7
DeltaN = 0.001
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
res = get_mu_distortion_plus([fd,DeltaN,mpd,med,alphad])
print(res)


fd = 0.01
epsilon = 1e-7
DeltaN = 0.01
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
res = get_mu_distortion_plus([fd,DeltaN,mpd,med,alphad])
print(res)

fd = 0.01
epsilon = 1e-7
DeltaN = 0.001
mpd = 0.01 * GeV
med = 0.001 * GeV
alphad = 0.05
res = get_mu_distortion_plus([fd,DeltaN,mpd,med,alphad])
print(res)

(7.9505816821784e-06, 132515.93320160368, 114052.5068657962, 113484.58330564831, 1.117741569972558e-15, 1.0013600218288697e-16, 9.287946392758926e-17, 93.24579216429294, 11.277245225341355, 10.564972108531762)
(1.1644705300019576e-05, 70112.8305053364, 61188.51069011471, 60923.183945708646, 3.0199012120590044e-16, 2.9902175741138505e-17, 2.790224979427082e-17, 89.99388977318952, 11.699723588185275, 11.012515244295288)


ValueError: f(a) and f(b) must have different signs

In [ ]:
#We're going to do some 1D scans in each parameter, away from a fiducial parameter point, to understand the dependence of the mu distortion on the model parameters
mu_cobefiras = 4.7 * 1e-5


zd_ls_1 = []
mu_ls_1 = []
epsilon_ls_1 = []
fd_ls = np.geomspace(0.001,1.0,20)
for fd in fd_ls:
    epsilon = 1e-7
    DeltaN = 0.001
    mpd = 0.01 * GeV
    med = 0.0001 * GeV
    alphad = 0.05
    mu = get_mu_distortion([fd,DeltaN,mpd,med,alphad])
    epsilon_limit = np.sqrt(mu_cobefiras/mu) * epsilon
    zd = find_zres(1)
    zd_ls_1.append(zd)
    mu_ls_1.append(mu)
    epsilon_ls_1.append(epsilon_limit)

zd_ls_2 = []
mu_ls_2 = []
epsilon_ls_2 = []
DeltaN_ls = np.geomspace(0.001,1.0,20)
for DeltaN in DeltaN_ls:
    epsilon = 1e-7
    fd = 0.01
    #DeltaN = 0.001
    mpd = 0.01 * GeV
    med = 0.0001 * GeV
    alphad = 0.05
    mu = get_mu_distortion([fd,DeltaN,mpd,med,alphad])
    epsilon_limit = np.sqrt(mu_cobefiras/mu) * epsilon
    zd = find_zres(1)
    zd_ls_2.append(zd)
    mu_ls_2.append(mu)
    epsilon_ls_2.append(epsilon_limit)

zd_ls_3 = []
mu_ls_3 = []
epsilon_ls_3 = []
mpd_ls =  np.geomspace(0.01,0.8,20) * GeV
for mpd in mpd_ls:
    #print(mpd/GeV)
    epsilon = 1e-7
    fd = 0.01
    DeltaN = 0.001
    #mpd = 0.01 * GeV
    med = 0.0001 * GeV
    alphad = 0.05
    mu = get_mu_distortion([fd,DeltaN,mpd,med,alphad])
    epsilon_limit = np.sqrt(mu_cobefiras/mu) * epsilon
    zd = find_zres(1)
    zd_ls_3.append(zd)
    mu_ls_3.append(mu)
    epsilon_ls_3.append(epsilon_limit)

zd_ls_4 = []
mu_ls_4 = []
epsilon_ls_4 = []
med_ls = np.geomspace(0.00001,0.001,20) * GeV
for med in med_ls:
    epsilon = 1e-7
    fd = 0.01
    DeltaN = 0.001
    mpd = 0.01 * GeV
    #med = 0.0001 * GeV
    alphad = 0.05
    mu = get_mu_distortion([fd,DeltaN,mpd,med,alphad])
    epsilon_limit = np.sqrt(mu_cobefiras/mu) * epsilon
    zd = find_zres(1)
    zd_ls_4.append(zd)
    mu_ls_4.append(mu)
    epsilon_ls_4.append(epsilon_limit)

zd_ls_5 = []
mu_ls_5 = []
epsilon_ls_5 = []
alphad_ls =  np.geomspace(0.012,0.05,20)
for alphad in alphad_ls:
    #print(alphad)
    epsilon = 1e-7
    fd = 0.01
    DeltaN = 0.001
    mpd = 0.01 * GeV
    med = 0.0001 * GeV
    #alphad = 0.05
    mu = get_mu_distortion([fd,DeltaN,mpd,med,alphad])
    epsilon_limit = np.sqrt(mu_cobefiras/mu) * epsilon
    zd = find_zres(1)
    zd_ls_5.append(zd)
    mu_ls_5.append(mu)
    epsilon_ls_5.append(epsilon_limit)

In [ ]:
mu_cobefiras = 4.7 * 1e-5

epsilon = 1
fd = 0.01
DeltaN = 0.001
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = get_mu_distortion([fd,DeltaN,mpd,med,alphad])
epsilon_limit = np.sqrt(mu_cobefiras/mu) * epsilon
print(mu)
print(epsilon_limit)

In [ ]:
fig=plt.figure(figsize=(4,4),dpi=200)
plt.plot(fd_ls,epsilon_ls_1)
plt.loglog()
plt.xlabel(r'$f_{D}$',fontsize=16)
plt.ylabel(r'$\epsilon$',fontsize=16)
plt.title('Upper limit on $\epsilon$ vs $f_{D}$')
plt.savefig('fd_epsilon_limits_20250623.png',bbox_inches='tight')

fig=plt.figure(figsize=(4,4),dpi=200)
plt.plot(fd_ls,zd_ls_1)
plt.loglog()
plt.xlabel(r'$f_{D}$',fontsize=16)
plt.ylabel(r'$z_{res}$',fontsize=16)
plt.title('$z_{res}$ vs $f_{D}$')
plt.savefig('fd_zres_20250623.png',bbox_inches='tight')


In [ ]:


plt.figure(figsize=(4,4),dpi=200)
plt.plot(DeltaN_ls,epsilon_ls_2)
plt.loglog()
plt.xlabel(r'$\Delta N$',fontsize=16)
plt.ylabel(r'$\epsilon$',fontsize=16)
plt.title('Upper limit on $\epsilon$ vs $\Delta N$')
plt.savefig('DeltaN_epsilon_limits_20250623.png',bbox_inches='tight')


plt.figure(figsize=(4,4),dpi=200)
plt.plot(DeltaN_ls,zd_ls_2)
plt.loglog()
plt.xlabel(r'$\Delta N$',fontsize=16)
plt.ylabel(r'$z_{res}$',fontsize=16)
plt.title('$z_{res}$ vs $\Delta N$')
plt.savefig('DeltaN_zres_20250623.png',bbox_inches='tight')


In [ ]:

plt.figure(figsize=(4,4),dpi=200)
plt.plot(mpd_ls/GeV,epsilon_ls_3)
plt.semilogx()
plt.xlabel(r'$m_{p_{D}}$ [GeV]',fontsize=16)
plt.ylabel(r'$\epsilon$',fontsize=16)
plt.title('Upper limit on $\epsilon$ vs $m_{p_{D}}$')
plt.savefig('mpd_epsilon_limits_20250623.png',bbox_inches='tight')


plt.figure(figsize=(4,4),dpi=200)
plt.plot(mpd_ls/GeV,zd_ls_3)
plt.loglog()
plt.xlabel(r'$m_{p_{D}}$ [GeV]',fontsize=16)
plt.ylabel('$z_{res}$',fontsize=16)
plt.title('$z_{res}$ vs $m_{p_{D}}$')
plt.savefig('mpd_zres_20250623.png',bbox_inches='tight')


In [ ]:


plt.figure(figsize=(4,4),dpi=200)
plt.plot(med_ls/GeV,epsilon_ls_4)
plt.loglog()
plt.xlabel(r'$m_{e_{D}}$ [GeV]',fontsize=16)
plt.ylabel(r'$\epsilon$',fontsize=16)
plt.title('Upper limit on $\epsilon$ vs $m_{e_{D}}$')
plt.savefig('med_epsilon_limits_20250623.png',bbox_inches='tight')


plt.figure(figsize=(4,4),dpi=200)
plt.plot(med_ls/GeV,zd_ls_4)
plt.loglog()
plt.xlabel(r'$m_{e_{D}}$ [GeV]',fontsize=16)
plt.ylabel('$z_{res}$',fontsize=16)
plt.title('$z_{res}$ vs $m_{e_{D}}$')
plt.savefig('med_zres_20250623.png',bbox_inches='tight')



In [ ]:


plt.figure(figsize=(4,4),dpi=200)
plt.plot(alphad_ls,epsilon_ls_5)
#plt.loglog()
plt.xlabel(r'$\alpha_{D}$',fontsize=16)
plt.ylabel(r'$\epsilon$',fontsize=16)
plt.title(r'Upper limit on $\epsilon$ vs $\alpha_{D}$')
plt.savefig('alphad_epsilon_limits_20250623.png',bbox_inches='tight')


plt.figure(figsize=(4,4),dpi=200)
plt.plot(alphad_ls,zd_ls_5)
plt.semilogy()
plt.xlabel(r'$\alpha_{D}$',fontsize=16)
plt.ylabel('$z_{res}$',fontsize=16)
plt.title(r'$z_{res}$ vs $\alpha_{D}$')
plt.savefig('alphad_zres_20250623.png',bbox_inches='tight')


In [ ]:
params = np.load('/gpfs/projects/EssigGroup/jbarron/adm_spectral_distortions/MuDistortionParameters_20250622.npy')
values = np.load('/gpfs/projects/EssigGroup/jbarron/adm_spectral_distortions/MuDistortionValues_20250622.npy')
mu_cobefiras = 4.7 * 1e-5
epsilon_ref = 1e-7
epsilon_limit = np.sqrt(mu_cobefiras/values) * epsilon_ref

xid_ls = ((7/8) * (11/4)**(-4/3) * params[:,1])**(1/4)
mu_ls = params[:,2] * params[:,3]/(params[:,2] + params[:,3])
Bd_ls = params[:,4]**2 * mu_ls/2
Td0_ls = xid_ls * T0
z_recomb_ls = 0.01 * Bd_ls/Td0_ls

#Already done
# z_res_ls = []
# dlnmgddlnz_ls = []
# for paramlist in params:
#     set_global_params(paramlist)
#     zres = find_zres(1)
#     z_res_ls.append(zres)
#     dlnmgddlnz_ls.append(dlnmgddlnz(zres,1))
z_res_ls = np.load('/gpfs/home/jbarron/aDM_spectraldistortions/data/MuDistortionZres_20250623.npy')
dlnmgddlnz_ls = np.load('/gpfs/home/jbarron/aDM_spectraldistortions/data/MuDistortion_dlnmgddlnz_20250623.npy')

In [ ]:
z_recomb_ls = 0.02 * Bd_ls/Td0_ls
plt.hist(z_res_ls/z_recomb_ls,bins=np.geomspace(0.1,10,100));
plt.semilogx()

In [ ]:
z_res_ls

In [ ]:
plt.scatter(z_recomb_ls,z_res_ls)
plt.loglog()
plt.xlabel('z recomb')
plt.ylabel('z res')
from scipy.stats import linregress
obj = linregress(Bd_ls/Td0_ls,z_res_ls)

In [ ]:
#np.save('/gpfs/home/jbarron/aDM_spectraldistortions/data/MuDistortionLimits_20250623.npy',epsilon_limit)

In [ ]:
#np.save('/gpfs/home/jbarron/aDM_spectraldistortions/data/MuDistortionZres_20250623.npy',z_res_ls)

In [ ]:
#np.save('/gpfs/home/jbarron/aDM_spectraldistortions/data/MuDistortion_dlnmgddlnz_20250623.npy',dlnmgddlnz_ls)

In [ ]:
plt.hist(dlnmgddlnz_ls,bins=np.geomspace(1,1e5,100));
plt.semilogx()

In [ ]:
plt.figure(figsize=(4,4),dpi=200)
plt.hist2d(z_res_ls,epsilon_limit,bins=(np.geomspace(1e3,2e7,100),np.geomspace(1e-9,1e-5,100)),cmin=1);
plt.loglog()
plt.xlabel('$z_{res}$ with $x_D$=1')
plt.ylabel('Upper limit on $\epsilon$')
plt.colorbar()

In [ ]:
plt.figure(figsize=(4,4),dpi=200)
plt.hist2d(dlnmgddlnz_ls,epsilon_limit,bins=(np.geomspace(1,1e5,100),np.geomspace(1e-9,1e-5,100)),cmin=1);
plt.loglog()
plt.xlabel('$dln(m_{\gamma})/dln(z)$ at $z_{res}$ with $x_D$=1')
plt.ylabel('Upper limit on $\epsilon$')
plt.colorbar()

In [ ]:
epsilon_limit

In [ ]:
epsilon_limit.shape

In [ ]:
plt.hist(epsilon_limit,bins=np.geomspace(1e-9,1e-5,100));
plt.xlabel('Upper bound on $\epsilon$',fontsize=16)
plt.semilogx()

In [ ]:
#The values I got out are (1e-7)^2 * Something. 
#I want (epsilon_limit)^2 * Something = mu_cobefiras
#So Something = values / (1e-7)^2. 
#epsilon_limit = sqrt(mu_cobefiras/something)
# = sqrt(mu_cobefiras/values) * 1e-7

# Code checks. Are we computing z_resonance correctly? Is Saha equilibrium a good approximation for x_eD(z) compared to CLASS? Does the resonance satisfy the conditions we require (i.e. redshift range?) Are our analytic functions for derivatives of eg photon masses etc correct, i.e. do they match the numerical derivative you get by evaluating the functions themselves? 

In [ ]:

#Comparing xd values
fd = 1
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e4,1e6,1000)

plt.figure(figsize=(4,4),dpi=200)

plt.plot(zrange,mgd2(zrange,10),linestyle='dashed',color='C1')
plt.plot(zrange,mgd2(zrange,1),linestyle='dashed',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),linestyle='dashed',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),linestyle='dashed',color='C4')

l1=Line2D([0], [0],label=r'$f_{D}=1$',linestyle='dashed',color='black',lw=3)
l2=Line2D([0], [0],label=r'$f_{D}=0.001$',linestyle='solid',color='black',lw=3)
plt.gca().add_line(l1)
plt.gca().add_line(l2)

#Comparing xd values
fd = 0.001
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e4,1e6,1000)
plt.plot(zrange,mg2(zrange),label='SM',color='C0')
plt.plot(zrange,mgd2(zrange,10),label=r'$x_{D}=10$',color='C1')
plt.plot(zrange,mgd2(zrange,1),label=r'$x_{D}=1$',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),label=r'$x_{D}=0.1$',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),label=r'$x_{D}=0.01$',color='C4')

#plt.text(6e4,1e-20,r'$f_{D}=0.001$'+'\n'+ r'$\Delta N=0.001$'+'\n'+ r'$m_{p_{D}}=0.01\ \mathrm{ GeV}$'+'\n'+ r'$m_{e_{D}}=0.0001\ \mathrm{ GeV}$'+'\n'+r'$\alpha_{D}=0.05$')
plt.legend(loc='upper left')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:

#Comparing xd values
fd = 0.01
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e4,1e6,1000)

plt.figure(figsize=(4,4),dpi=200)

plt.plot(zrange,mgd2(zrange,10),linestyle='dashed',color='C1')
plt.plot(zrange,mgd2(zrange,1),linestyle='dashed',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),linestyle='dashed',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),linestyle='dashed',color='C4')

l1=Line2D([0], [0],label=r'$\Delta N_{D}=0.001$',linestyle='dashed',color='black',lw=3)
l2=Line2D([0], [0],label=r'$\Delta N_{D}=0.01$',linestyle='solid',color='black',lw=3)
plt.gca().add_line(l1)
plt.gca().add_line(l2)

#Comparing xd values
fd = 0.01
DeltaN = 0.01
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e4,1e6,1000)
plt.plot(zrange,mg2(zrange),label='SM',color='C0')
plt.plot(zrange,mgd2(zrange,10),label=r'$x_{D}=10$',color='C1')
plt.plot(zrange,mgd2(zrange,1),label=r'$x_{D}=1$',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),label=r'$x_{D}=0.1$',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),label=r'$x_{D}=0.01$',color='C4')

#plt.text(6e4,1e-20,r'$f_{D}=0.001$'+'\n'+ r'$\Delta N=0.001$'+'\n'+ r'$m_{p_{D}}=0.01\ \mathrm{ GeV}$'+'\n'+ r'$m_{e_{D}}=0.0001\ \mathrm{ GeV}$'+'\n'+r'$\alpha_{D}=0.05$')
plt.legend(loc='upper left')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:

#Comparing xd values
fd = 0.01
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(5e4,1e6,1000)

plt.figure(figsize=(4,4),dpi=200)

plt.plot(zrange,mgd2(zrange,10),linestyle='dashed',color='C1')
plt.plot(zrange,mgd2(zrange,1),linestyle='dashed',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),linestyle='dashed',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),linestyle='dashed',color='C4')

l1=Line2D([0], [0],label=r'$m_{p_{D}}=0.01$ GeV',linestyle='dashed',color='black',lw=3)
l2=Line2D([0], [0],label=r'$m_{p_{D}}=0.1$ GeV',linestyle='solid',color='black',lw=3)
plt.gca().add_line(l1)
plt.gca().add_line(l2)

#Comparing xd values
fd = 0.01
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.1 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(5e4,1e6,1000)
plt.plot(zrange,mg2(zrange),label='SM',color='C0')
plt.plot(zrange,mgd2(zrange,10),label=r'$x_{D}=10$',color='C1')
plt.plot(zrange,mgd2(zrange,1),label=r'$x_{D}=1$',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),label=r'$x_{D}=0.1$',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),label=r'$x_{D}=0.01$',color='C4')

#plt.text(6e4,1e-20,r'$f_{D}=0.001$'+'\n'+ r'$\Delta N=0.001$'+'\n'+ r'$m_{p_{D}}=0.01\ \mathrm{ GeV}$'+'\n'+ r'$m_{e_{D}}=0.0001\ \mathrm{ GeV}$'+'\n'+r'$\alpha_{D}=0.05$')
plt.legend(loc='lower right')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:

#Comparing xd values
fd = 0.01
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e4,1e8,1000)

plt.figure(figsize=(4,4),dpi=200)

plt.plot(zrange,mgd2(zrange,10),linestyle='dashed',color='C1')
plt.plot(zrange,mgd2(zrange,1),linestyle='dashed',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),linestyle='dashed',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),linestyle='dashed',color='C4')

l1=Line2D([0], [0],label=r'$m_{e_{D}}=0.1$ MeV',linestyle='dashed',color='black',lw=3)
l2=Line2D([0], [0],label=r'$m_{e_{D}}=1$ MeV',linestyle='solid',color='black',lw=3)
plt.gca().add_line(l1)
plt.gca().add_line(l2)

#Comparing xd values
fd = 0.01
DeltaN = 0.001
mpd = 0.01 * GeV
med = 0.001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e4,1e8,1000)
plt.plot(zrange,mg2(zrange),label='SM',color='C0')
plt.plot(zrange,mgd2(zrange,10),label=r'$x_{D}=10$',color='C1')
plt.plot(zrange,mgd2(zrange,1),label=r'$x_{D}=1$',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),label=r'$x_{D}=0.1$',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),label=r'$x_{D}=0.01$',color='C4')

#plt.text(6e4,1e-20,r'$f_{D}=0.001$'+'\n'+ r'$\Delta N=0.001$'+'\n'+ r'$m_{p_{D}}=0.01\ \mathrm{ GeV}$'+'\n'+ r'$m_{e_{D}}=0.0001\ \mathrm{ GeV}$'+'\n'+r'$\alpha_{D}=0.05$')
plt.legend(loc='upper left')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:

#Comparing xd values
fd = 0.01
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e3,1e6,1000)

plt.figure(figsize=(4,4),dpi=200)

plt.plot(zrange,mgd2(zrange,10),linestyle='dashed',color='C1')
plt.plot(zrange,mgd2(zrange,1),linestyle='dashed',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),linestyle='dashed',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),linestyle='dashed',color='C4')

l1=Line2D([0], [0],label=r'$\alpha_{D}=0.05$',linestyle='dashed',color='black',lw=3)
l2=Line2D([0], [0],label=r'$\alpha_{D}=0.01$',linestyle='solid',color='black',lw=3)
plt.gca().add_line(l1)
plt.gca().add_line(l2)

#Comparing xd values
fd = 0.01
DeltaN = 0.001
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.01
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e3,1e6,1000)
plt.plot(zrange,mg2(zrange),label='SM',color='C0')
plt.plot(zrange,mgd2(zrange,10),label=r'$x_{D}=10$',color='C1')
plt.plot(zrange,mgd2(zrange,1),label=r'$x_{D}=1$',color='C2')
plt.plot(zrange,mgd2(zrange,0.1),label=r'$x_{D}=0.1$',color='C3')
plt.plot(zrange,mgd2(zrange,0.01),label=r'$x_{D}=0.01$',color='C4')

#plt.text(6e4,1e-20,r'$f_{D}=0.001$'+'\n'+ r'$\Delta N=0.001$'+'\n'+ r'$m_{p_{D}}=0.01\ \mathrm{ GeV}$'+'\n'+ r'$m_{e_{D}}=0.0001\ \mathrm{ GeV}$'+'\n'+r'$\alpha_{D}=0.05$')
plt.legend(loc='lower right')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:
Omegab/Omegadm

In [ ]:

fd = 0.183
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)

mpd = 0.938 * GeV
med = 0.000511 * GeV
alphad = 0.0073
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
zrange = np.geomspace(1e2,1e6,1000)

plt.figure(figsize=(4,4),dpi=200)
plt.plot(zrange,mgd2(zrange,1),label='1',color='C1')

fd = 0.183*2
DeltaN = 0.001
xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
mpd = 0.938 * GeV * 2 * 2
med = 0.000511 * GeV /2 / 2
alphad = 0.0073/2
mu = mpd * med/(mpd + med) 
Bd = alphad**2 * mu/2
Td0 = xid * T0
epsilon = 1e-7

#Critical density of the universe. 
rhocrit = 3 * H0**2 / (8 * np.pi * G) * eV**4 #This is in units of eV^4
#Planck 2018 values of dark matter and baryon energy density fractions today. Same as used in CLASS. 
Omegadm = 0.12038/h**2
Omegab = 0.022032/h**2
#Planck 2018 values of radiation and Lambda energy density fractions today. Same as used in CLASS. 
Omegar = 9.31734e-05 + 1.25113e-5 * DeltaN #This is photons + neutrinos + dark photons explicitly. 1.25113e-5 is the energy density of 1 neutrino species.
OmegaLambda = 0.68266
Omegam = Omegadm + Omegab

#Number densities of hydrogen and dark hydrogen today. 
n0 = rhocrit * Omegab / mp
nd0 = rhocrit * Omegadm * fd / (mpd + med - Bd)
plt.plot(zrange,mg2(zrange),label='SM',color='C0')
plt.plot(zrange,mgd2(zrange,1),label='2',color='C2',linestyle='dashed')

plt.legend(loc='lower right')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:
#Comparing xed(z) from Saha equilibrium with the xed(z) curve from CLASS. 
zrange = np.linspace(5e4,3e5,10000)
plt.axvline(find_zres(100),label='100',color='C0')
plt.axvline(find_zres(10),label='10',color='C1')
plt.axvline(find_zres(1),label='1',color='C2')
plt.axvline(find_zres(0.1),label='0.1',color='C3')
plt.axvline(find_zres(0.01),label='0.01',color='C4')
plt.legend(title='x_d')
plt.plot(zrange,xed_class(zrange),label='CLASS',color='black')
plt.plot(zrange,xed(zrange),label='Saha',color='red',linestyle='dashed')
plt.semilogy()
plt.legend()
plt.ylabel(r'$x_{ed}$',fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:
#Checking that the analytic derivative I've written for the photon mass and ionization fraction matches the numerical one. 
zrange = np.linspace(1e5,3e5,10000)

dmgd2_num = []

xd = 0.01
for z in zrange:
    rise = mgd2(z + zrange[1]-zrange[0],xd) - mgd2(z,xd) 
    run = zrange[1]-zrange[0]
    deriv = rise/run
    dmgd2_num .append(deriv)

plt.plot(zrange,dmgd2_num,color='black',label='numerical')
plt.plot(zrange,dmgd2(zrange,xd),color='red',linestyle='dashed',label='analytic')
plt.legend()
plt.loglog()
plt.ylabel(r'$dm^{2}_{gamma,d}/dz$',fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:
#Comparing xd values
zrange = np.geomspace(4e4,3e5,1000)
plt.plot(zrange,mg2(zrange),label='SM')
plt.plot(zrange,mgd2(zrange,50),label='50')
plt.plot(zrange,mgd2(zrange,10),label='10')
plt.plot(zrange,mgd2(zrange,1),label='1')
plt.plot(zrange,mgd2(zrange,0.1),label='0.1')
plt.plot(zrange,mgd2(zrange,0.01),label='0.01')

plt.plot(zrange,mgd2(zrange,0.001),label='0.001')
#plt.axvline(0.01 * Bd/Td0)
#plt.axvline(0.05 * Bd/Td0)
plt.legend(title='x_d')
plt.loglog()
plt.ylabel("Effective photon mass [eV]",fontsize=16)
plt.xlabel('z',fontsize=16)

In [ ]:
#Plot the rho and N integrands to understand what x contributes the most


fd = 1
DeltaN = 0.001
#xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.05
set_global_params([fd,DeltaN,mpd,med,alphad])
def rho_integrand(x):
    return x**2 * (1/(np.exp(x) - 1) - 1/(np.exp(x/xid) - 1)) * gammacon(x/xid)
def gammacon(xd): 
    z = find_zres(xd)
    return (np.pi * epsilon**2 * mg2(z))/(3 * Tcmb(z) * H(z))/np.abs(1 - ((1 + z)/(3 * mg2(z))) * dmgd2(z,xd))

x = np.geomspace(0.1,100,10)
#y = [rho_integrand(xx) for xx in x]
y = [quad(N_integrand,0,xx)[0] for xx in x]
plt.plot(x,y)
plt.semilogx()

In [ ]:
def gammacon(xd): 
    z = find_zres(xd)
    return (np.pi * epsilon**2 * mg2(z))/(3 * Tcmb(z) * H(z))/np.abs(1 - ((1 + z)/(3 * mg2(z))) * dmgd2(z,xd))


x = np.geomspace(0.0001,200,1000)
y = [gammacon(xx) for xx in x]
plt.axvline(0.2)
#plt.plot(x,y)
plt.plot(x,z)
plt.loglog()

In [ ]:
x = np.geomspace(0.0001,1000,1000)
z = [find_zres(xx) for xx in x]
DM = [dmgd2(zz,xx) for zz,xx in zip(z,x)]
plt.plot(x,DM)
plt.loglog()

In [ ]:
plt.plot(x,z)
plt.semilogx()

In [ ]:
def mg_wrapper(logz,xd):
    z = np.exp(logz)
    return mgd2(z,xd)- mg2(z)

#Function that uses bisection method to find the z where the dark photon mass and photon mass are equal. 
def find_zres(xd):
    return np.exp(bisect(mg_wrapper,np.log(0.01 * Bd/Td0),np.log(0.2 * Bd/Td0),args=(xd),disp=True))

A = np.linspace(np.log(0.01 * Bd/Td0),np.log(0.1 * Bd/Td0),100)
B1 = mg_wrapper(A,0.01)
B2 = mg_wrapper(A,0.1)
B3 = mg_wrapper(A,1)
B4 = mg_wrapper(A,100)

In [ ]:
y[10]

In [ ]:

fd = 0.01
DeltaN = 0.01
#xid = ((7/8) * (11/4)**(-4/3) * DeltaN)**(1/4)
mpd = 0.01 * GeV
med = 0.0001 * GeV
alphad = 0.04
set_global_params([fd,DeltaN,mpd,med,alphad])
x = np.linspace(0.1*Bd/Td0,0.22*Bd/Td0,1000,dtype=np.longdouble)
y = xed(x)
#plt.plot(x,(1-y)/y)
plt.plot(x,(1-y))

#plt.axvline(0.07*Bd/Td0)
plt.semilogx()

In [ ]:
A = np.geomspace(0.01 * Bd/Td0,0.2 * Bd/Td0,100)#np.exp(np.linspace(np.log(0.01 * Bd/Td0),np.log(0.2 * Bd/Td0),100))
B = mgd2(A,100)
plt.plot(A,B)
plt.axvline(0.07 * Bd/Td0)

In [ ]:
plt.plot(A,B1)
plt.plot(A,B2)
plt.plot(A,B3)
#plt.plot(A,B4)